# Tests du projet RAG, étape par étape
Lancez les cellules **de haut en bas**, une par une. Après un **Restart**, recommencez toujours par la cellule 0.

## 0. Préparation (à lancer en premier, à chaque fois)

In [1]:
import os, sys
# Se placer à la racine du projet, même après un Restart
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("Dossier de travail :", os.getcwd())

Dossier de travail : c:\Users\farah\Downloads\rag-contrats-bancaires\rag-contrats-bancaires


## 1. Séparer le texte source en un fichier par contrat
Mettez vos fichiers `.txt` (qui contiennent plusieurs contrats) dans `textes_source/`. Cette cellule crée `contrats/contrat_01.txt`, `contrat_02.txt`, etc. À lancer seulement quand le texte source change.

In [2]:
import re
from pathlib import Path

Path("contrats").mkdir(exist_ok=True)
# Reconnaît "Contrat n° 5", "## Contrat n° 5", "**Contrat n° 5**"...
motif = re.compile(r"(?=^[\s#*]*Contrat\s+n\s*[°º]?\s*\d+)", re.MULTILINE | re.IGNORECASE)

for source in Path("textes_source").glob("*.txt"):
    texte = source.read_text(encoding="utf-8")
    for partie in motif.split(texte):
        m = re.match(r"[\s#*]*Contrat\s+n\s*[°º]?\s*(\d+)", partie, re.IGNORECASE)
        if m:
            num = int(m.group(1))
            propre = partie.strip().lstrip("#* ")
            Path(f"contrats/contrat_{num:02d}.txt").write_text(propre, encoding="utf-8")
            print("Créé : contrat", num, "depuis", source.name)

print("Fichiers dans contrats :", len(list(Path("contrats").glob("*.txt"))))

Créé : contrat 1 depuis Crédits immobiliers.txt
Créé : contrat 2 depuis Crédits immobiliers.txt
Créé : contrat 3 depuis Crédits immobiliers.txt
Créé : contrat 4 depuis Crédits immobiliers.txt
Créé : contrat 5 depuis Crédits immobiliers.txt
Créé : contrat 6 depuis Crédits immobiliers.txt
Créé : contrat 7 depuis Crédits immobiliers.txt
Créé : contrat 8 depuis Crédits immobiliers.txt
Créé : contrat 9 depuis Crédits immobiliers.txt
Créé : contrat 10 depuis Crédits immobiliers.txt
Créé : contrat 11 depuis Crédits immobiliers.txt
Créé : contrat 12 depuis Crédits immobiliers.txt
Créé : contrat 13 depuis Crédits immobiliers.txt
Créé : contrat 14 depuis Crédits immobiliers.txt
Créé : contrat 15 depuis Crédits immobiliers.txt
Créé : contrat 16 depuis Crédits immobiliers.txt
Créé : contrat 17 depuis Crédits immobiliers.txt
Créé : contrat 18 depuis Crédits immobiliers.txt
Créé : contrat 19 depuis Crédits immobiliers.txt
Créé : contrat 20 depuis Crédits immobiliers.txt
Créé : contrat 21 depuis Créd

Si la cellule ci-dessus crée **0 fichier**, affichez le début du texte source pour voir le format des titres :

In [3]:
for source in Path("textes_source").glob("*.txt"):
    print("=====", source.name)
    print(source.read_text(encoding="utf-8")[:500])

===== Crédits immobiliers.txt
Crédits immobiliers (contrats 1 à 6)
Six prêts de 250 000 à 1 200 000 MAD, sur 12 à 300 mois, garantis par hypothèque sauf le prêt relais.
Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition
Réf. BMR-IMM-2026-00147 · Signé à Casablanca le 14 septembre 2026
Entre Banque Méridien du Royaume, SA au capital de 1 200 000 000 MAD, siège 112 boulevard d'Anfa, Casablanca, RC Casablanca 400117 (« le Prêteur »), et M. Youssef El Amrani, né le 3 mars 1986, cadre commercial, CIN BK482913, demeurant 


## 2. Extraction : le texte de chaque contrat est-il bien lu ?
Chaque contrat doit avoir plusieurs centaines de caractères.

In [4]:
from extraction import lire_dossier

contrats = lire_dossier("contrats")
for nom, texte in contrats.items():
    titre = texte.splitlines()[0] if texte else "(VIDE)"
    print(f"{len(texte):>6} caractères | {nom} | {titre}")

  1945 caractères | contrat_01 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition
  1999 caractères | contrat_02 | Contrat n° 2 — Prêt immobilier pour construction de villa
  1711 caractères | contrat_03 | Contrat n° 3 — Prêt immobilier à taux variable capé
  1374 caractères | contrat_04 | Contrat n° 4 — Prêt au logement social avec garantie publique
  1467 caractères | contrat_05 | Contrat n° 5 — Rachat de crédit immobilier
  1713 caractères | contrat_06 | Contrat n° 6 — Prêt relais remboursable in fine
  1505 caractères | contrat_07 | Contrat n° 7 — Prêt personnel non affecté
  1460 caractères | contrat_08 | Contrat n° 8 — Prêt personnel aux fonctionnaires avec prélèvement à la source
  1533 caractères | contrat_09 | Contrat n° 9 — Prêt affecté aux travaux d'aménagement
  1632 caractères | contrat_10 | Contrat n° 10 — Prêt études avec différé d'amortissement
  1332 caractères | contrat_11 | Contrat n° 11 — Crédit affecté à l'achat d'équipement ménager
  1382 caractères | c

## 3. Découpage : les articles sont-ils reconnus ?
On doit voir `préambule`, puis `Article 1, 2, 3...`. Si tout est en `préambule`, le motif des articles est à adapter dans `src/decoupage.py`.

In [5]:
from decoupage import decouper_contrat

nom, texte = next(iter(contrats.items()))
morceaux = decouper_contrat(nom, texte)
print(nom, ":", len(morceaux), "morceaux\n")
for m in morceaux:
    print("Article", m["metadata"]["article"], "|", m["texte"][:110])

contrat_01 : 7 morceaux

Article préambule | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Contrat n° 1 — Prêt immobilier à taux fixe pour 
Article 1 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 1 – Objet. Financement de l'acquisition d'u
Article 2 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 2 – Conditions financières.
• Montant : 850
Article 3 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 3 – Déblocage et garanties. Les fonds sont 
Article 4 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 4 – Remboursement anticipé. Total ou partie
Article 5 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 5 – Retard et défaillance. Les sommes impay
Article 6 | Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 6 – Réflexion et litiges. L'offre est maint


## 4. Embeddings : Ollama transforme-t-il un texte en vecteur ?
Avec `bge-m3`, on doit voir `1024` puis 5 nombres.

In [6]:
from indexation import embed

v = embed(["Remboursement anticipé"])[0]
print(len(v), v[:5])

1024 [-0.011853803, 0.050703812, 0.009477933, -0.025396131, -0.021527493]


## 5. Indexation : (re)construire la base vectorielle
Efface l'ancienne base puis indexe tous les contrats. À relancer après chaque modification des contrats ou du découpage.

In [8]:
import chromadb
from indexation import get_collection, indexer_contrat

client = chromadb.PersistentClient(path="base_vectorielle")
try:
    client.delete_collection("contrats")   # repartir d'une base propre
except Exception:
    pass

collection = get_collection()
for nom, texte in contrats.items():
    print(nom, ":", indexer_contrat(nom, texte, collection), "morceaux indexés")
print("\nTotal dans la base :", collection.count(), "morceaux")

contrat_01 : 7 morceaux indexés
contrat_02 : 7 morceaux indexés
contrat_03 : 7 morceaux indexés
contrat_04 : 7 morceaux indexés
contrat_05 : 7 morceaux indexés
contrat_06 : 7 morceaux indexés
contrat_07 : 7 morceaux indexés
contrat_08 : 7 morceaux indexés
contrat_09 : 7 morceaux indexés
contrat_10 : 7 morceaux indexés
contrat_11 : 7 morceaux indexés
contrat_12 : 7 morceaux indexés
contrat_13 : 7 morceaux indexés
contrat_14 : 7 morceaux indexés
contrat_15 : 7 morceaux indexés
contrat_16 : 7 morceaux indexés
contrat_17 : 7 morceaux indexés
contrat_18 : 7 morceaux indexés
contrat_19 : 7 morceaux indexés
contrat_20 : 7 morceaux indexés
contrat_21 : 7 morceaux indexés
contrat_22 : 7 morceaux indexés
contrat_23 : 7 morceaux indexés
contrat_24 : 7 morceaux indexés
contrat_25 : 7 morceaux indexés
contrat_26 : 7 morceaux indexés
contrat_27 : 7 morceaux indexés
contrat_28 : 7 morceaux indexés
contrat_29 : 7 morceaux indexés
contrat_30 : 7 morceaux indexés

Total dans la base : 210 morceaux


## 6. Recherche : quels articles sont trouvés pour une question ?
Vérifiez que la bonne réponse se trouve dans au moins un des extraits.

In [9]:
from rag import rechercher

question = "Quelle est l'indemnité de remboursement anticipé du rachat de crédit immobilier ?"
for p in rechercher(question):
    print(p["contrat"], "| article", p["article"])
    print("  ", p["texte"][:250], "\n")

contrat_05 | article 4
   Contrat n° 5 — Rachat de crédit immobilier | Art. 4 – Remboursement anticipé. Indemnité d'un mois d'intérêts sur le capital remboursé. 

contrat_01 | article 4
   Contrat n° 1 — Prêt immobilier à taux fixe pour acquisition | Art. 4 – Remboursement anticipé. Total ou partiel (minimum 10 % du capital restant dû), avec préavis de 30 jours. Indemnité : un mois d'intérêts sur le capital remboursé, dans la limite de 

contrat_25 | article 4
   Contrat n° 25 — Crédit de promotion immobilière | Art. 4 – Remboursement anticipé. Gratuit lorsqu'il provient du produit des ventes ; sinon, indemnité de 1 % du capital remboursé. 

contrat_02 | article 4
   Contrat n° 2 — Prêt immobilier pour construction de villa | Art. 4 – Remboursement anticipé. Gratuit dans la limite de 10 % du capital initial par année civile. Au-delà : indemnité d'un mois d'intérêts sur la part excédentaire. 



## 7. Réponse complète du RAG (recherche + Ollama)

In [10]:
from rag import repondre

resultat = repondre(question)
print(resultat["reponse"])
print("\nSources :", resultat["sources"])

Indemnité d'un mois d'intérêts sur le capital remboursé.

Référence : Contrat n° 5, Art. 4

Sources : [{'contrat': 'contrat_05', 'article': '4'}, {'contrat': 'contrat_01', 'article': '4'}, {'contrat': 'contrat_25', 'article': '4'}, {'contrat': 'contrat_02', 'article': '4'}]


Pour chercher **dans un seul contrat**, ajoutez son nom :

In [11]:
resultat = repondre("Quel est le TEG ?", contrat="contrat_05")
print(resultat["reponse"])

Le TEG est de 6,52 %. Selon l'article 2 du contrat n° 5.


## 8. Extraction des informations clés en JSON

In [12]:
from rag import extraire_infos
import json

infos = extraire_infos(contrats["contrat_05"] if "contrat_05" in contrats else texte)
print(json.dumps(infos, indent=2, ensure_ascii=False))

{
  "preteur": "Banque Zénith Maroc",
  "emprunteur": "Mme Nadia Fassi Fihri",
  "type_contrat": "Rachat de crédit immobilier",
  "montant_mad": 600000,
  "duree_mois": 180,
  "taux_nominal": 5.4,
  "teg": 6.52,
  "echeance_mad": 5203.71,
  "frais_dossier": 3500,
  "assurance": 0.32,
  "garanties": "Mainlevée de l'hypothèque antérieure et inscription d'une hypothèque de premier rang à hauteur de 720 000 MAD sur le titre foncier n° 55690/T",
  "remboursement_anticipe": "Indemnité d'un mois d'intérêts sur le capital remboursé."
}
